In [ ]:
pip install cartesia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 16.7 MB/s eta 0:00:00


In [8]:
import json
import os
from cartesia import Cartesia

# Initialize the client (Make sure your CARTESIA_API_KEY is in your environment variables)
client = Cartesia(api_key="sk_car_BsN7EN2Ctmv7C26JMsjq1U")

# Load your evaluation dataset
input_json = "hindi_evaluation_set.json"
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

output_dir = "sonic3_eval_audio"
os.makedirs(output_dir, exist_ok=True)

# Iterate through the JSON and generate audio for each Hindi entry
# Iterate through the dictionary's keys and nested dictionaries
for index, (category_key, nested_data) in enumerate(data.items()):

    # Grab the actual Hindi sentence from the "text" field
    text_to_speak = nested_data.get("text", "")

    if not text_to_speak:
        continue

    print(f"Generating audio for item {index} ({category_key})...")

    # Call the Sonic 3 API
    response = client.tts.generate(
        model_id="sonic-3",
        transcript=text_to_speak,
        voice={"id": "4877b818-c7fe-4c89-b1cf-eadf8e23da72"},
        language="hi",
        output_format={
            "container": "wav",
            "encoding": "pcm_s16le",
            "sample_rate": 44100
        }
    )

    # Save the output to a WAV file
    output_filename = os.path.join(output_dir, f"eval_output_{index}.wav")
    response.write_to_file(output_filename)

print("Sonic 3 TTS evaluation complete!")

Generating audio for item 0 (vowels_and_consonants)...
Generating audio for item 1 (velars_gutturals)...
Generating audio for item 2 (retroflexes)...
Generating audio for item 3 (palatals_and_nasals)...
Generating audio for item 4 (labials_and_aspirated)...
Generating audio for item 5 (loan_words_nukta)...
Generating audio for item 6 (complex_conjuncts)...
Generating audio for item 7 (dentals_and_visarga)...
Generating audio for item 8 (flaps_and_chandrabindu)...
Generating audio for item 9 (approximants_and_sibilants)...
Generating audio for item 10 (english_loan_words)...
Generating audio for item 11 (heavy_geminates)...
Generating audio for item 12 (ha_placement)...
Generating audio for item 13 (vowel_hiatus)...
Generating audio for item 14 (sanskrit_tatsama)...
Generating audio for item 15 (prosody_and_punctuation)...
Generating audio for item 16 (perso_arabic_nukta)...
Generating audio for item 17 (number_normalization)...
Generating audio for item 18 (consonant_clusters_r)...
Gen

In [9]:
!pip install openai-whisper jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 49.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=43672fbf5016ee9d5e42885250d59091761584ed3c0383461e27cab72df76990
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [10]:
import json
import os
import shutil
import whisper
from jiwer import wer, cer

# 1. Load the Whisper Medium model
print("Loading Whisper 'medium' model... (This may take a minute)")
model = whisper.load_model("medium")

# 2. Load your reference text from the JSON
input_json = "hindi_evaluation_set.json"
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

audio_dir = "sonic3_eval_audio"
references = []
hypotheses = []

print("\nStarting transcription and evaluation...")

# 3. Iterate through the JSON exactly like we did during generation
for index, (category_key, nested_data) in enumerate(data.items()):
    reference_text = nested_data.get("text", "")

    if not reference_text:
        continue

    audio_path = os.path.join(audio_dir, f"eval_output_{index}.wav")

    if not os.path.exists(audio_path):
        print(f"Skipping item {index} - Audio file not found.")
        continue

    # Transcribe the generated TTS audio back to text (Forcing Hindi language)
    print(f"Transcribing {audio_path}...")
    result = model.transcribe(audio_path, language="hi")
    hypothesis_text = result["text"].strip()

    # Save the texts for metric calculation
    references.append(reference_text)
    hypotheses.append(hypothesis_text)

    # Optional: Print them out to see how the model did
    print(f"Ref: {reference_text}")
    print(f"Hyp: {hypothesis_text}\n")

# 4. Calculate WER and CER
if references and hypotheses:
    overall_wer = wer(references, hypotheses)
    overall_cer = cer(references, hypotheses)

    print("-" * 30)
    print("🏆 EVALUATION RESULTS 🏆")
    print("-" * 30)
    print(f"Word Error Rate (WER):      {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
    print(f"Character Error Rate (CER): {overall_cer:.4f} ({overall_cer * 100:.2f}%)")
    print("-" * 30)
else:
    print("No audio files were evaluated. Check your directory.")

# 5. Zip the audio directory
zip_filename = "sonic3_eval_audio_backup"
print(f"\nZipping folder '{audio_dir}' into '{zip_filename}.zip'...")
shutil.make_archive(zip_filename, 'zip', audio_dir)

# 6. Trigger download (Specific to Google Colab / Jupyter)
try:
    from google.colab import files
    print("Initiating download...")
    files.download(f"{zip_filename}.zip")
except ImportError:
    print(f"Successfully created {zip_filename}.zip in your current directory.")
    print("Note: Automatic download trigger only works in Google Colab. Please download the file manually if running locally.")

Loading Whisper 'medium' model... (This may take a minute)


100%|██████████████████████████████████████| 1.42G/1.42G [00:07<00:00, 201MiB/s]



Starting transcription and evaluation...
Transcribing sonic3_eval_audio/eval_output_0.wav...
Ref: एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।
Hyp: एक आम आदमी और औरत इमली के पेर के निज़े बैट कर उन बुन रहें।

Transcribing sonic3_eval_audio/eval_output_1.wav...
Ref: कमल और काव्या ने खेत से कद्दू उखाड़ा, फिर गरम घी का घड़ा उठाकर घर की ओर भागे।
Hyp: कमल और काव्या ने खेच से कदू खाडा फिर गरंगी का घडा उठाकर गर की वर भागे

Transcribing sonic3_eval_audio/eval_output_2.wav...
Ref: षट्कोण के अंदर रखे ढक्कन और डमरू को देखकर ठग टमाटर टोकरी में डालकर डर गया।
Hyp: शटकोंड के अंदर रक्षे धक्कन और दम्रू को देखकर थब टमाटर तोकरी में डाल कर डर गया।

Transcribing sonic3_eval_audio/eval_output_3.wav...
Ref: चंचल छतरी लेकर झमाझम बारिश में जंगल की ओर चली गई।
Hyp: चंचल चट्री लेकर जमाजम भारिश में जंगल की रुज चली गई।

Transcribing sonic3_eval_audio/eval_output_4.wav...
Ref: भालू ने भारी पेड़ पर बैठकर मीठा फल खाया और पानी पी लिया।
Hyp: भालू ने भारी पेड़ पर बैट कर मीथा फल खाया और पानी पी लिया।

Tran

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>